# Uma introdução à Physical Informed Neural Network (PINN)

Este notebook tem como objetivo introduzir o conceito de *PINN* e suas principais aplicações.

**Autor:** Edélio Gabriel Magalhães de Jesus.

## Introdução

### **O problema de trabalhar somente com dados**

Já é amplamente reconhecido que o aprendizado de máquina transformou a forma como a ciência é conduzida. Modelos de *Machine Learning* são capazes de realizar tarefas de previsão e classificação com alta eficiência, especialmente quando dispõem de grandes volumes de dados. Em particular, redes neurais conseguem capturar relações não lineares, extraindo padrões diretamente dos dados, muitas vezes sem a necessidade de modelagem explícita.

No entanto, essa abordagem levanta uma limitação fundamental:

**e quando os dados são escassos, incompletos ou simplesmente impossíveis de obter?**

Considere, por exemplo, o problema de estimar a temperatura interna de um reator nuclear. A instrumentação direta em regiões críticas pode ser inviável por razões técnicas, econômicas ou de segurança. Como consequência, o conjunto de dados disponível é limitado e, frequentemente, insuficiente para treinar modelos puramente orientados por dados com confiabilidade.

Esse cenário é comum em diversas áreas da ciência e engenharia: sistemas físicos complexos, governados por leis bem estabelecidas, mas com observações experimentais restritas.

É nesse contexto que surgem as **Physics-Informed Neural Networks (PINNs)**.

<div style="text-align: center;">
  <img 
    src="https://www.mathworks.com/discovery/physics-informed-neural-networks/_jcr_content/mainParsys/columns/e4219b80-580a-4cc2-a14e-84b7087007c5/image.adapt.full.medium.png/1767870644981.png" 
    alt="Physics-Informed Neural Networks diagram"
    style="max-width: 600px; width: 100%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://www.mathworks.com/discovery/physics-informed-neural-networks.html' target='_blank'>
    MathWorks - Physics-Informed Neural Networks (PINNs)
  </a>
</div>


As PINNs integram conhecimento físico diretamente no processo de treinamento da rede neural, fazendo o modelo respeitar as leis fundamentais que governam o sistema.

---

Mas antes de partir para as PINN, vamos revisar os princiapis componentes e passos de uma rede neural clássica.

### **A rede neural clássica**

<div style="display: flex; justify-content: center; gap: 50px;">
  <img 
    src="https://miro.medium.com/v2/resize:fit:640/format:webp/1*L8-kTBfzIumpzJDW5yqZFQ.png"
    style="max-width: 300px; height: auto;"
  >
  <img 
    src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*JpfN1b4VkBUrZGdcHpAEfA.png"
    style="max-width: 300px; height: auto;"
  >
</div>

<div style="text-align: center; margin-top: 10px; font-size: 0.9em; color: #555;">
  Fonte: 
  <a href="https://medium.com/@anwarhermuche/introdu%C3%A7%C3%A3o-aos-algoritmos-de-classifica%C3%A7%C3%A3o-perceptron-fdb3f3c94627" target="_blank">
    Medium — Introdução aos algoritmos de classificação: Perceptron
  </a>
</div>

Uma rede neural clássica é composta por camadas de neurônios artificiais organizados em três grupos principais: a **camada de entrada**, que recebe os dados brutos; as **camadas ocultas** (*hidden layers*), onde ocorre o processamento e a extração de representações intermediárias; e a **camada de saída**, que produz a predição ou classificação final.

Cada neurônio realiza uma operação simples: computa uma combinação linear ponderada de suas entradas,

$$
z = \sum_{i} w_i x_i + b \tag{1}
$$

onde:

- $x_i$ são as **entradas** do neurônio — os valores recebidos da camada anterior (ou os dados brutos, na primeira camada);
- $w_i$ são os **pesos** (*weights*) — parâmetros ajustáveis que controlam a importância de cada entrada. Um peso grande amplifica a contribuição de $x_i$; um peso próximo de zero a suprime;
- $b$ é o **viés** (*bias*) — um parâmetro adicional que desloca a saída independentemente das entradas, funcionando como o coeficiente linear de uma reta. Sem ele, a superfície de decisão da rede seria sempre forçada a passar pela origem.

Em seguida, aplica-se sobre $z$ uma **função de ativação** não linear $\sigma(\cdot)$, produzindo a saída do neurônio:

$$
a = \sigma(z) = \sigma\!\left(\sum_{i} w_i x_i + b\right)
$$

A não-linearidade é essencial: sem ela, empilhar várias camadas seria equivalente a ter uma única camada — pois a composição de funções lineares continua sendo linear. É a função de ativação que quebra essa linearidade e permite que a rede represente funções complexas [[ref]](#funcao-ativacao).

O processo de treinamento segue quatro passos fundamentais:

1. **Forward pass** — os dados percorrem a rede da entrada à saída, gerando uma predição $\hat{y}$.
2. **Cálculo da perda** — uma função de perda $\mathcal{L}(\hat{y}, y)$ mede o quão longe a predição está do valor real $y$.
3. **Backpropagation** — o gradiente da perda em relação a cada peso é calculado via regra da cadeia, propagando o erro de volta pela rede.
4. **Atualização dos pesos** — um otimizador (como SGD ou Adam) ajusta os pesos na direção que reduz a perda:

$$
w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w} \tag{2}
$$

onde $\eta$ é a taxa de aprendizado.

Esse ciclo se repete por muitas épocas, até que a rede aprenda a mapear entradas em saídas com boa precisão — desde que haja dados suficientes para guiar esse processo. 

Mas então, qual foi a grande inovação implementada por RAISSI, PERDIKARIS e KARNIADAKIS em 2017 [[ref]](#original-paper) ?

### **O diferencial das PINNs**

O grande avanço está no passo 2: **Cálculo da perda**. Ao invés de simplesmente compararmos a previsão com os valores reais do *dataset*, incluímos as perdas relacionadas ao resíduo da equação diferencial e às condições de contorno e iniciais. O que os dados não forem capazes de ensinar à rede, a Física (ou Matemática, como achar melhor) ensina!  

#### ``A função de perda das PINNs``

A formulação original, proposta por Raissi et al. [[ref]](#original-paper), define a função de perda total como a soma de dois termos:

$$
L(\theta) = \text{MSE}_u + \text{MSE}_f \tag{3}
$$

O primeiro termo, $\text{MSE}_u$, penaliza a rede quando ela erra nos pontos onde a solução é **conhecida** — as condições de contorno (CC) e as condições iniciais (CI). O segundo, $\text{MSE}_f$, penaliza a rede quando ela **viola a equação diferencial**. Seguindo a notação de Baty [[ref]](#hands-on-paper), podemos reescrever isso de forma mais explícita:

$$
L(\theta) = \omega_{\text{data}}\, L_{\text{data}}(\theta) + \omega_{\text{PDE}}\, L_{\text{PDE}}(\theta) \tag{4}
$$

onde os pesos $\omega_{\text{data}},\, \omega_{\text{PDE}} \geq 0$ controlam a importância relativa de cada parcela e são hiperparâmetros a serem ajustados.

**A perda física** $L_{\text{PDE}}$ é avaliada em um conjunto de $N_c$ pontos $\{\mathbf{x}_i\}_{i=1}^{N_c}$ amostrados livremente no interior do domínio $\Omega$, chamados de **pontos de colocação** (*collocation points*). Esses pontos não precisam ter nenhum valor de solução associado — a rede simplesmente é obrigada a satisfazer $\mathcal{F} = 0$ ali:

$$
L_{\text{PDE}}(\theta) = \frac{1}{N_c} \sum_{i=1}^{N_c} \left| \mathcal{F}(u_\theta(\mathbf{x}_i)) \right|^2 \tag{5}
$$

onde $\mathcal{F}(\cdot) = 0$ é a EDP escrita na forma de resíduo para $\mathbf{x} \in \Omega$.

**A perda de dados** $L_{\text{data}}$ é avaliada nos **pontos de treinamento** (*training points*) — os $N_{\text{data}}$ pontos onde a solução é conhecida, localizados na fronteira $\partial\Omega$ (CC) e no instante inicial $t = 0$ (CI):

$$
L_{\text{data}}(\theta) = \frac{1}{N_{\text{data}}} \sum_{i=1}^{N_{\text{data}}} \left| u_\theta(\mathbf{x}_i^{\text{data}}) - u_i^{\text{data}} \right|^2 \tag{6}
$$

A mesma expressão cobre tanto CCs quanto CIs — a diferença está apenas em *onde* os pontos de treinamento estão localizados no domínio espaço-temporal.

#### ``O que são os "dados" de uma PINN?``

Aqui vale parar e pensar: de onde vêm os dados $u_i^{\text{data}}$?

Nas PINNs para problemas diretos, os únicos dados necessários são justamente os valores de CC e CI. Isso é o que torna as PINNs interessantes: **não é preciso um grande dataset de observações** — basta conhecer o comportamento da solução nas bordas do domínio e no instante inicial.

> **Exemplo:** imagine resolver a equação do calor em uma barra metálica. Você sabe que as pontas estão fixas a uma diferença $\delta T$ (CC) e conhece o perfil de temperatura inicial ao longo da barra (CI). Esses valores são seus *dados de treinamento*. Nos pontos de colocação, espalhados pelo interior do domínio espaço-temporal, você não sabe a temperatura — mas exige que a equação do calor seja satisfeita.

<div style="text-align: center;">
  <img 
    src="https://i0.wp.com/vamosestudarfisica.com/wp-content/uploads/2023/01/conducao-do-calor.jpg?w=409&ssl=1" 
    alt="Fluxo de calor em uma barra metálica"
    style="max-width: 600px; width: 80%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://vamosestudarfisica.com/como-se-processa-a-propagacao-do-calor/' target='_blank'>
    Vamos Estudar Física - Como se processa a propagação do calor?
  </a>
</div>

---
---

#### `EXTRA! (Um pequeno aprofundamento)`

#### **Vanilla-PINNs e Hard-PINNs**

> A formulação acima, referida por Baty [[ref]](#hands-on-paper) como ***vanilla-PINNs***, impõe as CCs e CIs de forma *soft* — como termos de penalização na função de perda. A rede é incentivada a respeitá-las, mas não é forçada de maneira exata.
>
> Existe uma alternativa: as ***hard-PINNs***, cuja ideia foi introduzida por Lagaris et al. (1998) [[ref]](#lagaris-paper). Nessa abordagem, as condições de contorno são embutidas diretamente na arquitetura da rede por meio de uma **função de tentativa** (*trial function*):
>
> $$
> u_\theta(\mathbf{x}) = A(\mathbf{x}) + B(\mathbf{x})\, u_\theta^*(\mathbf{x}) \tag{7}
> $$
>
> onde $A(\mathbf{x})$ satisfaz exatamente as CCs sem parâmetros ajustáveis, e $B(\mathbf{x})$ se anula na fronteira — de modo que a saída da rede $u_\theta^*$ não interfere nas condições de contorno. Com isso, o termo $L_{\text{data}}$ desaparece completamente e a função de perda se reduz a:
>
> $$
> L(\theta) = L_{\text{PDE}}(\theta) \tag{8}
> $$
>
> A vantagem é maior precisão nas fronteiras; a desvantagem é que construir $A$ e $B$ pode ser difícil ou mesmo impossível para geometrias complexas ou condições de Neumann.
>
> Faremos pelo menos um exemplo comparando as duas abordagens. 

---
---

#### ``E se eu tiver mais dados do que apenas CC e CI?``

Uma extensão natural é incluir, além das CCs e CIs, dados observados no **interior** do domínio — provenientes de uma solução analítica, numérica ou experimental. Esses pontos internos entram no mesmo $L_{\text{data}}$, e a estrutura da perda da Eq. (4) permanece inalterada.

Essa abordagem pode ajudar a convergência em regimes de difícil aprendizado — soluções com alta frequência, gradientes acentuados ou geometrias complexas. É importante notar, no entanto, que mesmo com dados internos abundantes, o termo $L_{\text{PDE}}$ **continua presente e necessário**: os dados complementam o aprendizado, mas não substituem a restrição física.

## Principais aplicações

### ``Problemas diretos e problemas inversos``

As PINNs se aplicam naturalmente a dois tipos de problemas, e é importante não confundi-los.

<div style="text-align: center;">
  <img 
    src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTU7Uq083Cuy8c89AF6eMvLEHzUAbXIK489CQ&s" 
    alt="Problema direto VS Problema Inverso"
    style="max-width: 400px; width: 50%; height: auto;"
  >
</div>

<div style='text-align: center; margin-top: 10px; font-size: 0.9em; color: #555'>
  Fonte:
  <a href='https://downloads.editoracientifica.com.br/articles/220709394.pdf' target='_blank'>
    Problemas Inversos: Conceitos Gerais e Aplicação Prática 
  </a>
</div>

**Problemas diretos** seguem o paradigma clássico da simulação numérica: conhecidos os parâmetros da EDP, as CCs e as CIs, determina-se a solução $u(\mathbf{x})$. Todo o desenvolvimento acima se enquadra aqui. Alguns exemplos práticos:

- **Transferência de calor** — dada a condutividade térmica do material e as temperaturas nas fronteiras, determinar o campo de temperatura ao longo do tempo [[ref]](#cai-heat);
- **Dinâmica de fluidos** — dadas as condições de contorno de velocidade e pressão, resolver as equações de Navier-Stokes para obter o campo de escoamento [[ref]](#raissi-ns);
- **Mecânica estrutural** — dados o módulo de elasticidade e as cargas aplicadas, determinar o campo de deslocamentos e tensões em uma estrutura [[ref]](#haghighat-solid).

**Problemas inversos** partem da direção oposta: algum parâmetro ou termo da própria EDP é desconhecido — um coeficiente de difusão, um termo fonte, uma viscosidade — e busca-se determiná-lo a partir de dados observados da solução. Nesses casos, o parâmetro desconhecido é tratado como uma variável adicional do processo de otimização, aprendida junto com os pesos $\theta$ da rede [[1]](#original-paper). Alguns exemplos:

- **Identificação de viscosidade** — a partir de medições esparsas do campo de velocidade de um fluido, determinar a viscosidade cinemática $\nu$ nas equações de Navier-Stokes [[ref]](#raissi-ns);
- **Caracterização de materiais** — a partir de dados de deformação medidos experimentalmente, identificar o módulo de elasticidade ou lei constitutiva de um material desconhecido [[ref]](#haghighat-solid);
- **Estimativa de parâmetros em epidemiologia** — a partir de dados observados de contágio, estimar as taxas de transmissão e recuperação em modelos do tipo SIR [[ref]](#dandekar-covid);

> **Atenção:** ter dados no interior do domínio **não define**, por si só, um problema inverso. O que o caracteriza é a existência de um termo desconhecido na EDP. Uma PINN pode receber dados internos em um problema direto — como na extensão híbrida descrita acima — sem que isso implique busca de parâmetros.

## Agora a jornada é sua!

Essa foi uma breve introdução sobre o conceito de *Physical Informed Neural Network*, em que destaquei a principal incrementação em relação às redes neurais clássicas: a função de perda modificada com termos da equação que descreve o problema tratado. A partir daqui, exploraremos alguns exemplos a fim de deixar cada vez mais claro como construir uma PINN e, pricipalmente, como treiná-la e testá-la. 

Você tem duas opções:

- Seguir a ordem indicada no README;
- Seguir a ordem que achar melhor.

Com exeção dos dois primeiros notebooks (``01_direct_stacionary_vannila.ipynb`` e ``02_direct_stationary_hard.ipynb``), não existe dependêmcia sequancial entre notebooks.
Confira o README para mais informações de como o repositório está organizado.

---

Dito isso, bons estudos!

## Referências

<a id="hands-on-paper"></a>  
BATY, Hubert. *A hands-on introduction to physics-informed neural networks for solving partial differential equations with benchmark tests taken from astrophysics and plasma physics*. arXiv preprint arXiv:2403.00599, 2024. Disponível em: https://arxiv.org/html/2403.00599v1#S2

<a id="cai-heat"></a>  
CAI, S.; WANG, Z.; WANG, S.; PERDIKARIS, P.; KARNIADAKIS, G. E. *Physics-informed neural networks for heat transfer problems*. Journal of Heat Transfer, v. 143, n. 6, 2021. Disponível em: https://asmedigitalcollection.asme.org/heattransfer/article/143/6/060801/1104439/Physics-Informed-Neural-Networks-for-Heat-Transfer

<a id="dandekar-covid"></a>  
DANDEKAR, R.; BARBASTATHIS, G. *Neural network aided quarantine control model estimation of COVID spread in Wuhan, China*. arXiv preprint arXiv:2004.02752, 2020. Disponível em: https://arxiv.org/abs/2004.02752

<a id="funcao-ativacao"></a>  
KOLLI, Aravind. *O Papel das Funções de Ativação em Redes Neurais: Um Guia Abrangente*. Disponível em: https://aravindkolli.medium.com/the-role-of-activation-functions-in-neural-networks-a-comprehensive-guide-2d481582122a

<a id="lagaris-paper"></a>  
LAGARIS, I. E.; LIKAS, A.; FOTIADIS, D. I. *Artificial neural networks for solving ordinary and partial differential equations*. IEEE Transactions on Neural Networks, v. 9, n. 5, p. 987–1000, 1998. Disponível em: https://ieeexplore.ieee.org/abstract/document/712178

<a id="original-paper"></a>  
RAISSI, Maziar; PERDIKARIS, Paris; KARNIADAKIS, George Em. *Physics informed deep learning (part i): Data-driven solutions of nonlinear partial differential equations*. arXiv preprint arXiv:1711.10561, 2017. Disponível em: https://arxiv.org/abs/1711.10561

<a id="raissi-ns"></a>  
RAISSI, M.; YAZDANI, A.; KARNIADAKIS, G. E. *Hidden fluid mechanics: Learning velocity and pressure fields from flow visualizations*. Science, v. 367, n. 6481, p. 1026–1030, 2020. Disponível em: https://www.science.org/doi/10.1126/science.aaw4741

<a id="haghighat-solid"></a>  
HAGHIGHAT, E.; RAISSI, M.; MOURE, A.; GOMEZ, H.; JUANES, R. *A physics-informed deep learning framework for inversion and surrogate modeling in solid mechanics*. Computer Methods in Applied Mechanics and Engineering, v. 379, p. 113741, 2021. Disponível em: https://www.sciencedirect.com/science/article/pii/S0045782521000773